## Refine Pipeline

This pipeline is responsible for enhancing the data for posterior datavis or any kind of statistical analysis.



In [0]:
import os
from pyspark.sql.types import StructType, StructField, StringType
from typing import Optional
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

garbage_collector_silver_lakehouse = "/Volumes/workspace/default/garabage_collection_volume/silver/garbage_collection_silver_lakehouse.delta/"

In [0]:
def load(df: DataFrame, table_dir: str):
    df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"workspace.default.{table_dir}")

In [0]:
def avg_collectors_per_month(df_garbage_collector: DataFrame) -> DataFrame:    
  df = df_garbage_collector
  df = df.withColumn("ANO/MES", F.concat(F.col("ANO"), F.lit("/"), F.col("MES")))
  df = df.groupBy("ANO/MES").agg(F.avg("COLETORES").alias("MEDIA_COLETORES")).orderBy("ANO/MES")
  df = df.withColumn("MEDIA_COLETORES", F.round(df.MEDIA_COLETORES, 2))
  return df


In [0]:
def sum_mass_per_month(df_garbage_collector: DataFrame) -> DataFrame:
    df = df_garbage_collector
    df = df.withColumn("ANO/MES", F.concat(F.col("ANO"), F.lit("/"), F.col("MES")))
    df = df.groupBy("ANO/MES").agg(F.sum("MASSA").alias("TOTAL_LIXO_TONELADAS")).orderBy("ANO/MES")
    df = df.withColumn("TOTAL_LIXO_TONELADAS", F.round(df.TOTAL_LIXO_TONELADAS, 2))
    return df

In [0]:
def indicators_per_region(df_garbage_collector: DataFrame) -> DataFrame: 
    rolled = (
    df_garbage_collector.rollup("REGIONAL")
        .agg(
            F.sum("VIAGENS").alias("TOTAL_VIAGENS"),
            F.sum("PERCURSO").alias("TOTAL_PERCURSO_KM"),
            F.sum("HORAS").alias("TOTAL_HORAS")))
    df = (rolled.withColumn("REGIONAL",
        F.when(F.col("REGIONAL").isNull(), F.lit("**TOTAL**")).otherwise(F.col("REGIONAL")))
        .orderBy(F.when(F.col("REGIONAL") == "**TOTAL**", 1).otherwise(0), F.col("REGIONAL")))
    return df


In [0]:
# Extraction
df_garbage_collector = spark.read.format("delta").load(garbage_collector_silver_lakehouse)

# Transform
df_avg_collectors_per_month = avg_collectors_per_month(df_garbage_collector)
df_sum_mass_per_month = sum_mass_per_month(df_garbage_collector)
df_indicators_per_region = indicators_per_region(df_garbage_collector)

# Load
load(df_avg_collectors_per_month, "gold_garbage_collection_avg_collectors_per_month")
load(df_sum_mass_per_month, "gold_garbage_collection_sum_mass_per_month")
load(df_indicators_per_region, "gold_garbage_collection_indicators_per_region")